# News To Stock Analyser 데이터준비

뉴스기사를 전달하면, 긍/부정분석 뿐아니라, 특정주식에 대한 긍/부정평가 처리 RAG 구현

In [1]:
%pip install -U datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

## 데이터준비
https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko

In [3]:
from datasets import load_dataset

dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')
dataset # train / valid / test

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [4]:
# 필터링 : train / category = 'economy'인 것만 남김
economy_dataset = dataset['train'].filter(lambda row: row['category'] == 'economy')
print(len(economy_dataset))

17088


In [5]:
import pandas as pd

df = economy_dataset.to_pandas() # Dataset ->
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


In [6]:
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate

class StockAnalysis(BaseModel):
    stock_related: bool = Field(description="뉴스와 주식 종목간의 연관성 여부")
    summary: str = Field(description='뉴스 요약')

    positive_stocks: str = Field(description='긍정적인 영향이 예상되는 주식 종목명 목록')
    positive_keywords: str = Field(description='긍정적인 영향의 근거가 되는 키워드 목록')
    positive_reasons: str = Field(description='긍정적인 영향이 예상되는 이유')

    summary: str = Field(description='부정적인 영향이 예상된느 주식 종목명 목록')
    summary: str = Field(description='부정적인 영향의 근거가 되는 키워드 목록')
    summary: str = Field(description='부정적인 영향이 예상되는 이유')

system_prompt = '''  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''

user_prompt = '''  # 분석 대상 뉴스 본문을 전달하는 사용자 프롬프트
다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.

[news]
{news}
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', user_prompt),
])

news = df['document'][0]
prompt.invoke({'news': news})

ChatPromptValue(messages=[SystemMessage(content="  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content='  # 분석 대상 뉴스 본문을 전달하는 사용자 프롬프트\n다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.\n\n[news]\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등

In [7]:
# Langchain 구성 및 구조화 출력
from langchain.chat_models import init_chat_model

llm = init_chat_model('gpt-5.6-luna')
chain = prompt | llm.with_structured_output(StockAnalysis) # 프롬프트 -> LLM -> StockAnalysis 구조화 출력

def analyze_news(news):
    return chain.invoke({'news': news})

analyze_news(news)

StockAnalysis(stock_related=True, summary='정부는 하반기 수출 증가세를 유지하고 무역수지를 개선하기 위해 수출 중소·중견기업 대상 무역금융을 연초 목표보다 40조 원 늘린 301조 원까지 확대한다. 또한 물류비 지원, 월 4척 이상의 임시선박 투입, 중소기업 전용 선복 확대, 해외 전시회 참가 지원 등을 추진한다. 반도체 등 첨단산업 육성 전략과 에너지 효율화 대책도 마련해 수출 경쟁력과 무역수지 개선을 지원할 계획이다. 다만 원자재 가격 상승과 높은 해상운임, 상반기 103억 달러의 무역수지 적자는 여전히 부담 요인이다.', positive_stocks='HMM, CJ대한통운, 삼성전자, SK하이닉스, LX인터내셔널', positive_keywords='무역금융 301조 원 확대, 임시선박 월 4척 이상 투입, 중소기업 전용 선복 확대, 물류비 지원, 반도체 등 첨단산업 육성, 수출 확대', positive_reasons='HMM은 임시선박 투입과 선복 확대 정책으로 해상운송 수요 및 정책 협력 기회가 늘어날 수 있다. CJ대한통운은 수출 중소기업의 물류비 지원과 수출 물동량 증가에 따라 국제물류 및 포워딩 사업 확대가 기대된다. 삼성전자와 SK하이닉스는 정부의 반도체 등 첨단산업 육성 전략과 수출 지원의 직접적인 수혜 가능성이 있다. LX인터내셔널은 수출입 물류 및 원자재·무역 관련 사업을 영위해 교역량 회복과 무역금융 확대의 간접 수혜가 가능하다.')

In [8]:
news = df['document'][100]
display(news)
print()

analyze_news(news)

'해수부 5일 개정 공유수면 관리 및 매립에 관한 법률 시행 헤럴드경제 홍태화 기자 앞으로 공유수면관리청이 어업·환경 등에 영향을 미칠 것으로 예상되는 공유수면 점용·사용 허가를 할 때 미리 어업인 등 이해관계자들의 의견을 들어야 한다. 해양수산부는 5일 이같은 내용이 담긴 개정 공유수면 관리 및 매립에 관한 법률과 같은 법 시행령·시행규칙이 이날부터 시행된다고 밝혔다. 바다·바닷가·하천 등 공유수면은 공유재이기 때문에 이를 점용·사용하기 위해서는 별도의 허가를 받아야 한다. 최근 해상풍력 발전시설 해변을 이용한 관광시설 등 대규모 시설이 공유수면을 장기적으로 점용·사용하는 경우가 늘어났지만 이해 관계자의 의견을 사전에 수렴할 수 없는 문제가 있었다. 이에 공유수면 점용·사용으로 인한 사회적 갈등이 증가했다. 이러한 문제를 해결하기 위해 해수부는 지난 1월 공유수면 점용·사용 허가를 할 때 이해관계자의 의견을 듣도록 공유수면 관리 및 매립에 관한 법률을 개정했다. 법 개정에 따라 공유수면관리청이 해양환경·수산자원·자연경관 보호 등에 영향을 끼칠 수 있는 공유수면 점용·사용 신청을 받은 경우 이를 관보 공보 와 인터넷 홈페이지에 공고해야 한다. 또 점용·사용 허가를 했을 때 피해를 볼 것으로 예상되는 어업인에 대한 의견 조사도 별도로 진행해야 한다. 황준성 해수부 해양공간정책과장은 공유수면 점용·사용으로 인한 이해 관계자의 피해를 방지하려는 법령 개정의 취지를 달성할 수 있도록 각 공유수면관리청과 협력해 관련 제도의 차질 없는 운영을 지원하겠다 고 말했다.'

StockAnalysis(stock_related=True, summary='해양수산부가 공유수면 점용·사용 허가 과정에서 어업인 등 이해관계자의 사전 의견을 의무적으로 듣도록 개정 법령을 시행했다. 해상풍력 발전시설과 해변 관광시설 등 대규모·장기 점용 사업은 관보·공보·홈페이지 공고와 피해 예상 어업인 의견조사를 거쳐야 하므로, 사업의 투명성과 갈등 조정에는 긍정적이지만 인허가 기간 연장과 사업 지연 가능성이 커질 수 있다.', positive_stocks='', positive_keywords='', positive_reasons='직접적인 수혜 상장 종목은 확인하기 어렵다. 장기적으로는 이해관계자 갈등과 사업 중단 위험을 낮출 수 있지만, 단기적으로는 신규 사업의 인허가 절차를 추가하는 규제이므로 관련 기업의 즉각적인 실적 개선으로 연결되기는 어렵다도 긍정적 수혜 종목은 제한적이다。? no')

In [9]:
df = df[:1000]
df['content'] = df['title'] + '\n' + df['document']

pd.set_option('display.max_colwidth', None)
df['content'].head()

0                                                                                                                                                                      추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 

In [10]:
# 증류방식 활용 (GPT 답변을 로컬LLM에게 학습)
from tqdm.auto import tqdm

results = []

for content in tqdm(df['content']):
    result = analyze_news(content)
    results.append(result)

df['result'] = results
df.head

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Pydantic(StockAnalysis) 결과를 JSON 문자열로 파싱
def parse_to_json(obj):
    return obj.model_dump_json()

df['result_json'] = df['result'].apply(parse_to_json)
df.head()


pydantic.BaseModel.model_dump_json() -> dict
pydantic.BaseModel.model_dump_json() -> json_str

In [ ]:
# result_json 컬럼 결측치 제거 / 인덱스 재정렬
df = df.dropna(subset=['result_json'])
df = df.reset_index(drop = True)

In [ ]:
df['system'] = system_prompt

df = df.rename(columns={
    'news': 'user',
    'result_json': 'assistant'
})

df = df[
    ['system', 'user', 'assistant']
]

df.head()

In [ ]:
df[['system','user','assistant']].to_json(
    'train.json',
    orient = 'recoeds',
    force_ascii = False,
    indent = 4
)

In [ ]:
import os
from datasets import Dataset

dataset = Dataset.from_pandas(df[['system','user','assistant']])
dataset.push_to_hub(
    'Genus-Jae/naver-economy-news2stock',
    token = os.environ['HF_TOKEN']
)